# Lab 2.2 - Train the Same Problem With PyTorch (MNIST)

This notebook has `# TODO` markers (TODO 1-7). Work through them in order -
later cells depend on earlier ones (e.g. the hyperparameter sweep in Step 2
reuses the `build_model()` function you write in Step 1).

MNIST downloads automatically the first time you run this notebook (via
`torchvision.datasets.MNIST`) - no manual download needed, but you do need a
working internet connection the first time.

In [ ]:
import time
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets
import matplotlib.pyplot as plt

torch.manual_seed(10)
np.random.seed(10)

print("PyTorch version:", torch.__version__)

## Step 1 - Build and train a small CNN

Architecture (matches the lab diagram):

`Input 1x28x28 -> Conv2d(8, 3x3, ReLU) -> MaxPool2d(2x2) -> Conv2d(16, 3x3, ReLU) -> MaxPool2d(2x2) -> Flatten -> Linear(32, ReLU) -> Linear(10)`

Kept small on purpose - this exact model gets quantized (2.3) and pruned +
deployed to the Pico (2.4).

Data loading and normalization are provided for you below.

In [ ]:
train_dataset = datasets.MNIST(root="./data", train=True, download=True)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True)

# Normalize to [0, 1] and add the channel dimension Conv2d expects (NCHW format: N x 1 x 28 x 28)
x_train = (train_dataset.data.numpy() / 255.0).astype("float32")[:, np.newaxis, :, :]
y_train = train_dataset.targets.numpy()
x_test  = (test_dataset.data.numpy()  / 255.0).astype("float32")[:, np.newaxis, :, :]
y_test  = test_dataset.targets.numpy()

print("x_train:", x_train.shape, " x_test:", x_test.shape)

**TODO 1:** Complete `build_model()`. It should return an `nn.Sequential`
model with two conv+pool blocks feeding a small linear head,
matching the architecture above. The function takes some hyperparameters as
arguments - you'll reuse this same function for the experiments in Step 2,
so make sure each argument actually controls the thing its name says it does.

A couple of other things worth knowing before you write this:
- PyTorch Conv2d expects inputs with shape `(N, C, H, W)`. After Conv2d(1, 8, 3) -> MaxPool2d(2x2) ->
  Conv2d(8, 16, 3) -> MaxPool2d(2x2) -> Flatten, the feature dimension into Linear is `16 * 5 * 5 = 400`.
- If `dropout > 0`, add an `nn.Dropout(dropout)` layer after the dense/Linear layer;
  if it's 0, skip it entirely.
- In PyTorch, `nn.CrossEntropyLoss` combines LogSoftmax and NLLLoss internally,
  so the output layer is `nn.Linear(dense_units, 10)` (logits, no Softmax layer).

In [ ]:
def build_model(num_filters1=8, num_filters2=16, dense_units=32,
                 activation="relu", dropout=0.0):
    # TODO 1: build and return the nn.Sequential model described above
    layers = []
    return nn.Sequential(*layers)

model = build_model()
print(model)

**TODO 2:** Train the model for 5 epochs, batch size 64, using `x_test`/`y_test` as validation data.
- Optimizer: `optim.Adam` (lr=0.001)
- Loss: `nn.CrossEntropyLoss()` (labels are integers 0-9, not one-hot)
- Metric: accuracy
Record training and validation accuracy in a dictionary `history` (with `"accuracy"` and
`"val_accuracy"` lists) for plotting in the next cell.

In [ ]:
# TODO 2: train the model, capturing accuracy and val_accuracy in history
history = None


In [ ]:
model.eval()
criterion = nn.CrossEntropyLoss()
with torch.no_grad():
    test_out = model(torch.from_numpy(x_test))
    test_loss = criterion(test_out, torch.from_numpy(y_test).long()).item()
    test_acc = (test_out.argmax(dim=1).numpy() == y_test).mean()

print(f"Baseline model -- test accuracy: {test_acc:.4f}, test loss: {test_loss:.4f}")

plt.figure(figsize=(6, 4))
train_curve = history.history["accuracy"] if hasattr(history, "history") else history["accuracy"]
val_curve = history.history["val_accuracy"] if hasattr(history, "history") else history["val_accuracy"]
plt.plot(train_curve, label="train accuracy")
plt.plot(val_curve, label="val accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Baseline CNN training curve")
plt.legend()
plt.tight_layout()
plt.savefig("baseline_training_curve.png", dpi=110)
plt.show()

## Step 2 - Experiment with training parameters

Try changing each of the following, one at a time, and note the effect on
accuracy, training time, and model size:

- Learning rate
- Optimizer (SGD vs Adam vs RMSprop)
- Number of epochs
- Batch size
- Activation functions
- Number of filters/neurons per layer
- Dropout
- Number of layers

**TODO 3:** Complete `run_experiment()` below (build, train, and evaluate a
model, timing the training run and counting its trainable parameters), then
add your own entries to the `results` list - at least one experiment per
bullet point above. Reuse `build_model()` and `run_experiment()`; don't
rewrite the training loop by hand for each one.

In [ ]:
import pandas as pd

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def run_experiment(name, optimizer="adam", epochs=3, batch_size=64, activation="relu",
                    num_filters1=8, num_filters2=16, dense_units=32, dropout=0.0, lr=0.001):
    # TODO 3a: build the model with the given hyperparameters
    m = None

    # TODO 3b: configure optimizer (same loss/metric as Step 1)

    # TODO 3c: time the training run with the given epochs/batch_size
    start = time.time()
    elapsed = None

    # TODO 3d: evaluate on the test set and count trainable params
    acc = None
    n_params = None

    return {
        "experiment": name,
        "test_accuracy": round(acc, 4) if acc is not None else None,
        "train_time_sec": round(elapsed, 1) if elapsed is not None else None,
        "trainable_params": n_params,
    }

# TODO 3e: add at least one run_experiment(...) call per bullet point above
results = []
results.append(run_experiment("baseline (adam, lr=default)", optimizer="adam"))

results_df = pd.DataFrame(results)
results_df

In [ ]:
plt.figure(figsize=(9, 4))
plt.barh(results_df["experiment"], results_df["test_accuracy"])
plt.xlabel("Test accuracy")
plt.title("Hyperparameter sweep -- test accuracy by configuration")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("hyperparameter_sweep.png", dpi=110)
plt.show()

best_row = results_df.loc[results_df["test_accuracy"].idxmax()]
print("Best configuration in this sweep:")
print(best_row)

## Step 3 - Create and test your own dataset

1. Draw all 10 digits (phone app, or on paper + photo).
2. Save them into the `my_digits/` folder as `digit_0.png`, `digit_1.png`,
   ... `digit_9.png` (one file per digit, filename matches the digit drawn).
3. Complete the preprocessing function below so your photos match MNIST's
   format **exactly**:
   - Convert to grayscale.
   - Resize to 28x28.
   - MNIST digits are **white strokes on a black background** - a normal
     photo of pen-on-paper is the opposite (dark strokes on a light
     background), so you'll need to invert it.
   - Normalize to [0, 1].

In [ ]:
from PIL import Image

def load_and_preprocess_digit(path):
    # TODO 4: grayscale -> resize to 28x28 -> invert -> normalize to [0,1]
    arr = None
    return arr

digits_dir = "my_digits"
my_x, my_y = [], []
for digit in range(10):
    path = os.path.join(digits_dir, f"digit_{digit}.png")
    my_x.append(load_and_preprocess_digit(path))
    my_y.append(digit)

my_x = np.array(my_x)[:, np.newaxis, :, :]  # NCHW format: (10, 1, 28, 28)
my_y = np.array(my_y)

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(my_x[i, 0, :, :], cmap="gray")
    ax.set_title(str(my_y[i]))
    ax.axis("off")
plt.tight_layout()
plt.savefig("my_digits_preview.png", dpi=110)
plt.show()

**TODO 5:** Evaluate your trained model on your own digits and print
whether each prediction was correct. Expect worse accuracy than on MNIST's
test set - be ready to explain why (distribution shift: your handwriting,
pen, and camera/scanner don't match how MNIST's ~70,000 images were
collected).

In [ ]:
# TODO 5: evaluate model on (my_x, my_y), then print true vs. predicted
# digit for each of the 10 images, with an OK/WRONG marker
my_acc = None
print(f"Accuracy on my own digits: {my_acc:.4f}  (MNIST test accuracy was {test_acc:.4f})")

## Step 4 - Save the trained model

**TODO 6:** Save the model state dict (`torch.save(model.state_dict(), "mnist_cnn.pth")`).
You'll load this exact model again in Lab 2.3 and Lab 2.4, so keep the file safe.

In [ ]:
# TODO 6: save the model state dict as mnist_cnn.pth


## Discussion Topics

Answer in a few sentences each:

1. Which single hyperparameter gave you the biggest accuracy gain (or loss)?
2. Where did your own handwritten digits fail, and why do you think that is?
3. What's the tradeoff between model size and accuracy in your experiments?